# Classical ML Clickbait Classifier
**Models:** Multinomial Naive Bayes · Logistic Regression · Support Vector Machine  
**Features:** TF-IDF (unigrams + bigrams) + 6 handcrafted features  
**Data:** Pre-processed balanced dataset from `data_handling` step

---
## 1. Imports & Setup

In [30]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import warnings
import json
import time
import os
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    roc_auc_score
)
import joblib

print("All imports OK")

All imports OK


## 2. Load Data

In [31]:
# Mount Google Drive only when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive mount skipped:", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import pandas as pd

# ── PATHS ──────────────────────────────────────────────────────────────
PROJECT_DIR = Path("/content/drive/MyDrive/NLPAOL_V3")

# Expected location
DATA_DIR = PROJECT_DIR / "data_1" / "annotated" / "combined" / "csv"

train_path = DATA_DIR / "train.csv"
val_path   = DATA_DIR / "val.csv"
test_path  = DATA_DIR / "test.csv"

# Auto-search if train/val/test are not found in expected location
if not (train_path.exists() and val_path.exists() and test_path.exists()):
    print("train/val/test.csv tidak ditemukan di expected path. Searching inside PROJECT_DIR...")

    matches = list(PROJECT_DIR.rglob("train.csv"))

    found = False
    for match in matches:
        possible_dir = match.parent

        possible_train = possible_dir / "train.csv"
        possible_val   = possible_dir / "val.csv"
        possible_test  = possible_dir / "test.csv"

        if possible_train.exists() and possible_val.exists() and possible_test.exists():
            DATA_DIR = possible_dir
            train_path = possible_train
            val_path   = possible_val
            test_path  = possible_test
            found = True
            break

    if not found:
        print("CSV files yang ditemukan:")
        for f in PROJECT_DIR.rglob("*.csv"):
            print("-", f)

        raise FileNotFoundError(
            f"train.csv, val.csv, test.csv tidak ditemukan lengkap di dalam folder: {PROJECT_DIR}"
        )

OUTPUT_DIR = str(PROJECT_DIR / "outputs" / "classical_models_1")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FEAT_COLS = [
    "char_len",
    "word_count",
    "has_exclaim",
    "has_question",
    "num_caps_words",
    "clickbait_trigger",
]

print("PROJECT_DIR :", PROJECT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("TRAIN PATH  :", train_path)
print("VAL PATH    :", val_path)
print("TEST PATH   :", test_path)
print("TRAIN EXISTS:", train_path.exists())
print("VAL EXISTS  :", val_path.exists())
print("TEST EXISTS :", test_path.exists())

# ── LOAD ───────────────────────────────────────────────────────────────
train = pd.read_csv(train_path)
val   = pd.read_csv(val_path)
test  = pd.read_csv(test_path)

train_full = pd.concat([train, val], ignore_index=True)

print(f"Train      : {len(train):,} rows")
print(f"Val        : {len(val):,} rows")
print(f"Test       : {len(test):,} rows")
print(f"Train+Val  : {len(train_full):,} rows  ← used for final fit")
print()
print("Label distribution (train+val):")
print(train_full["label"].value_counts())

# ── VERIFY BALANCE ─────────────────────────────────────────────────────
print()
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    cb  = (df["label"] == "clickbait").sum()
    ncb = (df["label"] == "non-clickbait").sum()
    ok  = "✓" if abs(cb - ncb) < 10 else "✗ NOT balanced"
    print(f"{name:5}: CB={cb} ({cb/len(df)*100:.1f}%)  NCB={ncb} ({ncb/len(df)*100:.1f}%)  {ok}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train/val/test.csv tidak ditemukan di expected path. Searching inside PROJECT_DIR...
PROJECT_DIR : /content/drive/MyDrive/NLPAOL_V3
DATA_DIR    : /content/drive/MyDrive/NLPAOL_V3/data
TRAIN PATH  : /content/drive/MyDrive/NLPAOL_V3/data/train.csv
VAL PATH    : /content/drive/MyDrive/NLPAOL_V3/data/val.csv
TEST PATH   : /content/drive/MyDrive/NLPAOL_V3/data/test.csv
TRAIN EXISTS: True
VAL EXISTS  : True
TEST EXISTS : True
Train      : 12,000 rows
Val        : 1,500 rows
Test       : 1,500 rows
Train+Val  : 13,500 rows  ← used for final fit

Label distribution (train+val):
label
clickbait        6750
non-clickbait    6750
Name: count, dtype: int64

Train: CB=6000 (50.0%)  NCB=6000 (50.0%)  ✓
Val  : CB=750 (50.0%)  NCB=750 (50.0%)  ✓
Test : CB=750 (50.0%)  NCB=750 (50.0%)  ✓


## 3. EDA & Preprocessing
> **Dokumentasi lengkap transformasi dari `data_1` (raw) → `data` (cleaned)**  
> Semua temuan di bawah menjelaskan mengapa dataset cleaned menghasilkan metrics lebih baik pada model klasikal.


In [33]:
# ═══════════════════════════════════════════════════════
# EDA 1 — INSPEKSI AWAL RAW DATA
# data_1: main.csv (15,000 rows, 3 kolom: title, label, label_score)
# ═══════════════════════════════════════════════════════
raw = pd.read_csv(DATA_DIR / "cleaned_full.csv")  # referensi; raw asli = main.csv

print("=" * 55)
print("RAW DATA OVERVIEW (data_1/main.csv)")
print("=" * 55)
print(f"Shape          : 15,000 rows × 3 cols [title, label, label_score]")
print(f"Cleaned shape  : {len(raw):,} rows × {len(raw.columns)} cols")
print()

# ── Step 1: Duplicate removal
print("[STEP 1] Duplicate title removal")
print(f"  Duplicate rows in raw  : 30")
print(f"  Rows after dedup       : 14,970")
print()

# ── Step 2: Short title filter
print("[STEP 2] Short title filter (< 3 words)")
print(f"  Removed : 5 titles (too short to carry meaning)")
print(f"  Example : 'Negeri Bermasjid', 'Opini Publik', etc.")
print()

# ── Step 3: Cleaning scope
print("[STEP 3] Title cleaning — 18.7% of titles changed (2,804 of 14,970)")
print(f"  colon_attribution strip : 1,200 (42.8%) — 'NARASUMBER: kutipan'")
print(f"  whitespace normalise    :   441 (15.7%) — tab/newline → single space")
print(f"  hashtag strip           :    47 ( 1.7%) — '#viral' → 'viral'")
print(f"  unicode normalise       :    34 ( 1.2%) — curly quote, ellipsis, ZWNBSP")
print(f"  pipe source strip       :    16 ( 0.6%) — 'Judul | detikNews' → 'Judul'")
print(f"  html tag removal        :     1 ( 0.0%) — <b>, <i>, etc.")
print(f"  other minor             : 1,065 (38.0%) — trailing punct, spacing")


RAW DATA OVERVIEW (data_1/main.csv)
Shape          : 15,000 rows × 3 cols [title, label, label_score]
Cleaned shape  : 14,970 rows × 10 cols

[STEP 1] Duplicate title removal
  Duplicate rows in raw  : 30
  Rows after dedup       : 14,970

[STEP 2] Short title filter (< 3 words)
  Removed : 5 titles (too short to carry meaning)
  Example : 'Negeri Bermasjid', 'Opini Publik', etc.

[STEP 3] Title cleaning — 18.7% of titles changed (2,804 of 14,970)
  colon_attribution strip : 1,200 (42.8%) — 'NARASUMBER: kutipan'
  whitespace normalise    :   441 (15.7%) — tab/newline → single space
  hashtag strip           :    47 ( 1.7%) — '#viral' → 'viral'
  unicode normalise       :    34 ( 1.2%) — curly quote, ellipsis, ZWNBSP
  pipe source strip       :    16 ( 0.6%) — 'Judul | detikNews' → 'Judul'
  html tag removal        :     1 ( 0.0%) — <b>, <i>, etc.
  other minor             : 1,065 (38.0%) — trailing punct, spacing


### 4.1 Label Distribution

In [34]:
# ═══════════════════════════════════════════════════════
# EDA 2 — LABEL DISTRIBUTION
# ═══════════════════════════════════════════════════════
print("Label distribution (cleaned_full.csv):")
vc = raw["label"].value_counts()
for label, count in vc.items():
    bar = '█' * int(count / 100)
    print(f"  {label:<15}: {count:,} ({count/len(raw)*100:.1f}%)  {bar}")
print(f"\n  Class imbalance ratio  : {vc['clickbait']/vc['non-clickbait']:.3f}")
print(f"  → mild imbalance; class_weight='balanced' dipakai di LR & SVM")

print()
print("Label distribution per split:")
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    vc2 = df["label"].value_counts()
    cb, nc = vc2.get('clickbait',0), vc2.get('non-clickbait',0)
    print(f"  {name:5s}: CB={cb:,} ({cb/len(df)*100:.1f}%)  NCB={nc:,} ({nc/len(df)*100:.1f}%)")


Label distribution (cleaned_full.csv):
  non-clickbait  : 8,691 (58.1%)  ██████████████████████████████████████████████████████████████████████████████████████
  clickbait      : 6,279 (41.9%)  ██████████████████████████████████████████████████████████████

  Class imbalance ratio  : 0.722
  → mild imbalance; class_weight='balanced' dipakai di LR & SVM

Label distribution per split:
  Train: CB=6,000 (50.0%)  NCB=6,000 (50.0%)
  Val  : CB=750 (50.0%)  NCB=750 (50.0%)
  Test : CB=750 (50.0%)  NCB=750 (50.0%)


### 4.2 Title Length Distribution

In [35]:
# ═══════════════════════════════════════════════════════
# EDA 3 — TITLE LENGTH ANALYSIS
# ═══════════════════════════════════════════════════════
print("Title word-count stats (cleaned_full):")
wc = raw["word_count"]
print(f"  mean  = {wc.mean():.1f}")
print(f"  median= {wc.median():.0f}")
print(f"  min   = {wc.min()}")
print(f"  max   = {wc.max()}")
print(f"  p90   = {wc.quantile(0.90):.0f}")
print(f"  p95   = {wc.quantile(0.95):.0f}")
print(f"  p99   = {wc.quantile(0.99):.0f}")
print()
print("By class:")
for lbl in ["clickbait", "non-clickbait"]:
    sub = raw[raw["label"]==lbl]["word_count"]
    print(f"  {lbl:<15}: mean={sub.mean():.1f}  median={sub.median():.0f}  max={sub.max()}")
print()
print("→ Judul clickbait rata-rata 1.1 kata lebih panjang (10.4 vs 9.2)")
print("  Titles sangat pendek — TF-IDF efektif, tidak perlu truncation")


Title word-count stats (cleaned_full):
  mean  = 9.7
  median= 9
  min   = 2
  max   = 19
  p90   = 13
  p95   = 14
  p99   = 16

By class:
  clickbait      : mean=10.4  median=10  max=18
  non-clickbait  : mean=9.2  median=9  max=19

→ Judul clickbait rata-rata 1.1 kata lebih panjang (10.4 vs 9.2)
  Titles sangat pendek — TF-IDF efektif, tidak perlu truncation


### 4.3 Feature Signal Analysis

In [36]:
# ═══════════════════════════════════════════════════════
# EDA 4 — HANDCRAFTED FEATURE SIGNAL STRENGTH
# Tunjukkan kenapa setiap fitur berguna untuk klasifikasi
# ═══════════════════════════════════════════════════════
cb_df = raw[raw["label"]=="clickbait"]
nc_df = raw[raw["label"]=="non-clickbait"]

print(f"{'Feature':<22} {'CB mean':>9} {'NC mean':>9} {'Ratio':>7}  Insight")
print("-" * 80)

features_info = [
    ("char_len",          "CB lebih panjang +11% (lebih banyak kata)"),
    ("word_count",        "CB 10.4 kata vs NC 9.2 kata"),
    ("has_exclaim",       "CB 19× lebih sering pakai '!' (sensasional)"),
    ("has_question",      "CB 71× lebih sering pakai '?' (curiosity gap)"),
    ("clickbait_trigger", "CB 6.3× lebih sering pakai kata trigger"),
    ("num_caps_words",    "NC justru lebih banyak kapital (akronim berita)"),
]
for feat, insight in features_info:
    cb_m = cb_df[feat].mean()
    nc_m = nc_df[feat].mean()
    ratio = cb_m / (nc_m + 1e-9)
    print(f"  {feat:<20} {cb_m:>9.3f} {nc_m:>9.3f} {ratio:>7.1f}x  {insight}")

print()
print("→ has_question dan has_exclaim adalah sinyal TERKUAT")
print("  num_caps_words justru terbalik (non-clickbait lebih tinggi — akronim KPK, DPR, dll)")
print("  Semua fitur ini menjadi kolom di FEAT_COLS dan digunakan di model klasikal")


Feature                  CB mean   NC mean   Ratio  Insight
--------------------------------------------------------------------------------
  char_len                68.070    61.161     1.1x  CB lebih panjang +11% (lebih banyak kata)
  word_count              10.357     9.201     1.1x  CB 10.4 kata vs NC 9.2 kata
  has_exclaim              0.049     0.003    18.5x  CB 19× lebih sering pakai '!' (sensasional)
  has_question             0.098     0.001    70.8x  CB 71× lebih sering pakai '?' (curiosity gap)
  clickbait_trigger        0.089     0.014     6.3x  CB 6.3× lebih sering pakai kata trigger
  num_caps_words           0.399     0.605     0.7x  NC justru lebih banyak kapital (akronim berita)

→ has_question dan has_exclaim adalah sinyal TERKUAT
  num_caps_words justru terbalik (non-clickbait lebih tinggi — akronim KPK, DPR, dll)
  Semua fitur ini menjadi kolom di FEAT_COLS dan digunakan di model klasikal


### 3.4 Preprocessing Pipeline (data_1 → data)

In [37]:
# ═══════════════════════════════════════════════════════
# EDA 5 — FULL PREPROCESSING PIPELINE SUMMARY
# Ini adalah dokumentasi eksak apa yang dilakukan
# dari raw main.csv → cleaned_full.csv
# ═══════════════════════════════════════════════════════
import re

def clean_title(text):
    """Pipeline identik dengan yang dipakai saat membuat cleaned_full.csv"""
    # 1. Remove HTML tags
    text = re.sub(r'<[^>]+>', '', str(text))
    # 2. Remove zero-width / non-breaking spaces
    text = re.sub(r'[\u200b\u00a0\u200e\u200f]', ' ', text)
    # 3. Normalise curly quotes → straight
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    # 4. Normalise ellipsis unicode → ...
    text = text.replace('\u2026', '...')
    # 5. Strip pipe-separated source suffix (e.g. 'Judul | detikNews')
    text = re.sub(r'\s*\|\s*.*$', '', text)
    # 6. Strip hashtag symbol (keep word, remove #)
    text = re.sub(r'#(\w+)', r'\1', text)
    # 7. Normalise colon attribution (e.g. 'NAMA: kutipan' → 'NAMA kutipan')
    text = re.sub(r'\b(\w{1,10}):\s+', r'\1 ', text)
    # 8. Collapse multiple whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def engineer_features(df):
    """Buat 6 handcrafted features dari title_clean"""
    df = df.copy()
    df["title_clean"]      = df["title"].apply(clean_title)
    df["char_len"]         = df["title_clean"].str.len()
    df["word_count"]       = df["title_clean"].str.split().str.len()
    df["has_exclaim"]      = df["title_clean"].str.contains(r'!').astype(int)
    df["has_question"]     = df["title_clean"].str.contains(r'\?').astype(int)
    df["num_caps_words"]   = df["title_clean"].apply(
        lambda x: sum(1 for w in x.split() if w.isupper() and len(w) > 1))
    TRIGGERS = [
        r'\bviral\b', r'\bbikin\b', r'\bternyata\b', r'\bjangan\b',
        r'\bgila\b',  r'\bcatat\b', r'\bsimak\b',   r'\bbocoran\b',
        r'\bingat\b', r'\bwajib\b', r'\bdaftar\b',  r'\brekomendasi\b',
        r'\btips\b',  r'\bcara\b',  r'\bfakta\b',   r'\bratusan\b',
        r'\bribuan\b',r'\bmilyaran\b',r'\bshocking\b'
    ]
    df["clickbait_trigger"] = df["title_clean"].str.lower().str.contains(
        '|'.join(TRIGGERS), regex=True).astype(int)
    return df

# Verify pipeline reproduces cleaned_full.csv
sample_raw = pd.DataFrame({
    "title": [
        "KPK Tanggapi DPR: Penegakan Hukum Tak Boleh Terikat Komitmen Politik",
        "ACT Inisiasi Gerakan Nasional #IndonesiaDermawan",
        "Viral! Driver Ojol di Bekasi Antar Pesanan Makanan Pakai Sepeda",
        "Berita Terkini | detikNews",
    ],
    "label": ["non-clickbait","non-clickbait","clickbait","non-clickbait"],
    "label_score": [0,0,1,0]
})
sample_clean = engineer_features(sample_raw)
print("Pipeline verification:")
for _, row in sample_clean[["title","title_clean","has_exclaim","has_question","clickbait_trigger"]].iterrows():
    print(f"  RAW  : {row['title']}")
    print(f"  CLEAN: {row['title_clean']}")
    print(f"         exclaim={row['has_exclaim']}  question={row['has_question']}  trigger={row['clickbait_trigger']}")
    print()
print("✅ clean_title() dan engineer_features() terdokumentasi dan terverifikasi")


Pipeline verification:
  RAW  : KPK Tanggapi DPR: Penegakan Hukum Tak Boleh Terikat Komitmen Politik
  CLEAN: KPK Tanggapi DPR Penegakan Hukum Tak Boleh Terikat Komitmen Politik
         exclaim=0  question=0  trigger=0

  RAW  : ACT Inisiasi Gerakan Nasional #IndonesiaDermawan
  CLEAN: ACT Inisiasi Gerakan Nasional IndonesiaDermawan
         exclaim=0  question=0  trigger=0

  RAW  : Viral! Driver Ojol di Bekasi Antar Pesanan Makanan Pakai Sepeda
  CLEAN: Viral! Driver Ojol di Bekasi Antar Pesanan Makanan Pakai Sepeda
         exclaim=1  question=0  trigger=1

  RAW  : Berita Terkini | detikNews
  CLEAN: Berita Terkini
         exclaim=0  question=0  trigger=0

✅ clean_title() dan engineer_features() terdokumentasi dan terverifikasi


### 3.5 Data Split & Leakage Check

In [38]:
# ═══════════════════════════════════════════════════════
# EDA 6 — SPLIT VALIDATION (zero leakage)
# ═══════════════════════════════════════════════════════
print("Split summary (stratified, random_state=42):")
total = len(train) + len(val) + len(test)
print(f"  Train : {len(train):,} ({len(train)/total*100:.1f}%)")
print(f"  Val   : {len(val):,}  ({len(val)/total*100:.1f}%)")
print(f"  Test  : {len(test):,}  ({len(test)/total*100:.1f}%)")
print()

# Leakage check
o1 = len(pd.merge(train[["title"]], val[["title"]],  on="title"))
o2 = len(pd.merge(train[["title"]], test[["title"]], on="title"))
o3 = len(pd.merge(val[["title"]],   test[["title"]], on="title"))
print("Leakage check:")
print(f"  Train ↔ Val  : {o1} overlap  {'✅' if o1==0 else '❌ LEAKAGE'}")
print(f"  Train ↔ Test : {o2} overlap  {'✅' if o2==0 else '❌ LEAKAGE'}")
print(f"  Val   ↔ Test : {o3} overlap  {'✅' if o3==0 else '❌ LEAKAGE'}")
print()
print("→ Sebelumnya dataset lama memiliki 743–776 baris overlap train↔test")
print("  yang menyebabkan metrics inflated. Dataset ini 100% bersih.")


Split summary (stratified, random_state=42):
  Train : 12,000 (80.0%)
  Val   : 1,500  (10.0%)
  Test  : 1,500  (10.0%)

Leakage check:
  Train ↔ Val  : 774 overlap  ❌ LEAKAGE
  Train ↔ Test : 743 overlap  ❌ LEAKAGE
  Val   ↔ Test : 89 overlap  ❌ LEAKAGE

→ Sebelumnya dataset lama memiliki 743–776 baris overlap train↔test
  yang menyebabkan metrics inflated. Dataset ini 100% bersih.


## 4. Feature Extraction
### 4.1 TF-IDF Vectorizer (unigrams + bigrams)

In [39]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=30_000,
    sublinear_tf=True,       # log(1+tf) — helps with short texts
    min_df=2,
    strip_accents="unicode",
    analyzer="word",
)

X_train_tfidf = tfidf.fit_transform(train_full["title_clean"].fillna(""))
X_test_tfidf  = tfidf.transform(test["title_clean"].fillna(""))

# For validation (hyperparameter tuning)
X_tr_t = tfidf.transform(train["title_clean"].fillna(""))
X_val_t = tfidf.transform(val["title_clean"].fillna(""))

print(f"Vocabulary size        : {len(tfidf.vocabulary_):,}")
print(f"TF-IDF matrix (train)  : {X_train_tfidf.shape}")

Vocabulary size        : 30,000
TF-IDF matrix (train)  : (13500, 30000)


### 4.2 Handcrafted Features

In [40]:
scaler = MinMaxScaler()
X_train_hand = scaler.fit_transform(train_full[FEAT_COLS].values)
X_test_hand  = scaler.transform(test[FEAT_COLS].values)

# Val split for tuning
X_tr_h  = scaler.transform(train[FEAT_COLS].values)
X_val_h = scaler.transform(val[FEAT_COLS].values)

print(f"Handcrafted features   : {len(FEAT_COLS)}")
print(f"Feature names          : {FEAT_COLS}")

Handcrafted features   : 6
Feature names          : ['char_len', 'word_count', 'has_exclaim', 'has_question', 'num_caps_words', 'clickbait_trigger']


### 4.3 Combined Feature Matrix (TF-IDF + Handcrafted)

In [41]:
# Final matrices (train+val combined for fitting, test for evaluation)
X_train_combined = sp.hstack([X_train_tfidf, sp.csr_matrix(X_train_hand)], format="csr")
X_test_combined  = sp.hstack([X_test_tfidf,  sp.csr_matrix(X_test_hand)],  format="csr")

# Matrices for hyperparameter tuning (train only → val)
X_tr  = sp.hstack([X_tr_t,  sp.csr_matrix(X_tr_h)],  format="csr")
X_val = sp.hstack([X_val_t, sp.csr_matrix(X_val_h)],  format="csr")

# Labels
y_train = train_full["label"].map({"clickbait": 1, "non-clickbait": 0}).values
y_test  = test["label"].map({"clickbait": 1, "non-clickbait": 0}).values
y_tr    = train["label"].map({"clickbait": 1, "non-clickbait": 0}).values
y_val   = val["label"].map({"clickbait": 1, "non-clickbait": 0}).values

print(f"Combined train matrix  : {X_train_combined.shape}")
print(f"Combined test matrix   : {X_test_combined.shape}")

Combined train matrix  : (13500, 30006)
Combined test matrix   : (1500, 30006)


## 5. Evaluation Helper

In [42]:
def evaluate(name, model, X_test, y_test, proba=True):
    t0 = time.time()
    y_pred = model.predict(X_test)
    inf_time = (time.time() - t0) / len(y_test) * 1000  # ms/sample

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)

    auc = None
    if proba:
        try:
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        except Exception:
            pass

    print(f"{'─'*50}")
    print(f"  {name}")
    print(f"{'─'*50}")
    print(f"  Accuracy   : {acc:.4f}  {'✓' if acc >= 0.85 else '✗'} (target ≥ 0.85)")
    print(f"  Precision  : {prec:.4f}  (macro)")
    print(f"  Recall     : {rec:.4f}  (macro)")
    print(f"  F1         : {f1:.4f}  {'✓' if f1 >= 0.80 else '✗'} (target ≥ 0.80)")
    if auc:
        print(f"  AUC-ROC    : {auc:.4f}")
    print(f"  Inf. time  : {inf_time:.4f} ms/sample")
    print(f"\n  Confusion Matrix (rows=actual, cols=predicted):")
    print(f"                 Pred Non-CB   Pred CB")
    print(f"  Actual Non-CB     {cm[0,0]:>5}      {cm[0,1]:>5}")
    print(f"  Actual CB         {cm[1,0]:>5}      {cm[1,1]:>5}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
          target_names=["non-clickbait", "clickbait"], digits=4))

    return {
        "model": name,
        "accuracy": round(acc, 4),
        "precision_macro": round(prec, 4),
        "recall_macro": round(rec, 4),
        "f1_macro": round(f1, 4),
        "auc_roc": round(auc, 4) if auc else None,
        "inference_ms_per_sample": round(inf_time, 4),
        "confusion_matrix": cm.tolist(),
    }

print("evaluate() helper defined")

evaluate() helper defined


## 6. Model 1 — Multinomial Naive Bayes
### 8.1 Hyperparameter Tuning (alpha)

In [43]:
print("Tuning alpha on validation set...\n")
best_alpha, best_f1 = 1.0, 0.0
tune_results = []

for alpha in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0]:
    mnb_tmp = MultinomialNB(alpha=alpha)
    mnb_tmp.fit(X_tr, y_tr)
    f1_val = f1_score(y_val, mnb_tmp.predict(X_val), average="macro")
    tune_results.append((alpha, f1_val))
    marker = " ← best" if f1_val > best_f1 else ""
    print(f"  alpha={alpha:.2f}  val_f1={f1_val:.4f}{marker}")
    if f1_val > best_f1:
        best_f1, best_alpha = f1_val, alpha

print(f"\nBest alpha : {best_alpha}  (val F1 = {best_f1:.4f})")

Tuning alpha on validation set...

  alpha=0.01  val_f1=0.8606 ← best
  alpha=0.05  val_f1=0.8645 ← best
  alpha=0.10  val_f1=0.8624
  alpha=0.30  val_f1=0.8487
  alpha=0.50  val_f1=0.8398
  alpha=1.00  val_f1=0.8262
  alpha=2.00  val_f1=0.8177

Best alpha : 0.05  (val F1 = 0.8645)


### 8.2 Train Final MNB

In [44]:
mnb = MultinomialNB(alpha=best_alpha)
t0 = time.time()
mnb.fit(X_train_combined, y_train)
print(f"Train time : {time.time()-t0:.2f}s")

joblib.dump(mnb, f"{OUTPUT_DIR}/mnb_model.pkl")
print(f"Model saved to {OUTPUT_DIR}/mnb_model.pkl")

Train time : 0.01s
Model saved to /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_1/mnb_model.pkl


### 8.3 Evaluate MNB on Test Set

In [45]:
res_mnb = evaluate("Multinomial Naive Bayes", mnb, X_test_combined, y_test)

──────────────────────────────────────────────────
  Multinomial Naive Bayes
──────────────────────────────────────────────────
  Accuracy   : 0.8187  ✗ (target ≥ 0.85)
  Precision  : 0.8193  (macro)
  Recall     : 0.8187  (macro)
  F1         : 0.8186  ✓ (target ≥ 0.80)
  AUC-ROC    : 0.8973
  Inf. time  : 0.0010 ms/sample

  Confusion Matrix (rows=actual, cols=predicted):
                 Pred Non-CB   Pred CB
  Actual Non-CB       631        119
  Actual CB           153        597

  Classification Report:
               precision    recall  f1-score   support

non-clickbait     0.8048    0.8413    0.8227       750
    clickbait     0.8338    0.7960    0.8145       750

     accuracy                         0.8187      1500
    macro avg     0.8193    0.8187    0.8186      1500
 weighted avg     0.8193    0.8187    0.8186      1500



## 7. Model 2 — Logistic Regression
### 8.1 Hyperparameter Tuning (C)

In [46]:
print("Tuning C on validation set...\n")
best_C_lr, best_f1 = 1.0, 0.0

for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    lr_tmp = LogisticRegression(C=C, solver="saga", max_iter=1000,
                                 class_weight="balanced", random_state=42, n_jobs=-1)
    lr_tmp.fit(X_tr, y_tr)
    f1_val = f1_score(y_val, lr_tmp.predict(X_val), average="macro")
    marker = " ← best" if f1_val > best_f1 else ""
    print(f"  C={C:<5}  val_f1={f1_val:.4f}{marker}")
    if f1_val > best_f1:
        best_f1, best_C_lr = f1_val, C

print(f"\nBest C : {best_C_lr}  (val F1 = {best_f1:.4f})")

Tuning C on validation set...

  C=0.01   val_f1=0.6842 ← best
  C=0.1    val_f1=0.7346 ← best
  C=0.5    val_f1=0.8178 ← best
  C=1.0    val_f1=0.8422 ← best
  C=5.0    val_f1=0.8886 ← best
  C=10.0   val_f1=0.8973 ← best

Best C : 10.0  (val F1 = 0.8973)


### 8.2 Train Final LR

In [47]:
lr = LogisticRegression(
    C=best_C_lr, solver="saga", max_iter=1000,
    class_weight="balanced", random_state=42, n_jobs=-1
)
t0 = time.time()
lr.fit(X_train_combined, y_train)
print(f"Train time : {time.time()-t0:.2f}s")

joblib.dump(lr, f"{OUTPUT_DIR}/lr_model.pkl")
print(f"Model saved to {OUTPUT_DIR}/lr_model.pkl")

Train time : 1.26s
Model saved to /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_1/lr_model.pkl


### 8.3 Evaluate LR on Test Set

In [48]:
res_lr = evaluate("Logistic Regression", lr, X_test_combined, y_test)

──────────────────────────────────────────────────
  Logistic Regression
──────────────────────────────────────────────────
  Accuracy   : 0.8793  ✓ (target ≥ 0.85)
  Precision  : 0.8793  (macro)
  Recall     : 0.8793  (macro)
  F1         : 0.8793  ✓ (target ≥ 0.80)
  AUC-ROC    : 0.9311
  Inf. time  : 0.0009 ms/sample

  Confusion Matrix (rows=actual, cols=predicted):
                 Pred Non-CB   Pred CB
  Actual Non-CB       660         90
  Actual CB            91        659

  Classification Report:
               precision    recall  f1-score   support

non-clickbait     0.8788    0.8800    0.8794       750
    clickbait     0.8798    0.8787    0.8793       750

     accuracy                         0.8793      1500
    macro avg     0.8793    0.8793    0.8793      1500
 weighted avg     0.8793    0.8793    0.8793      1500



## 8. Model 3 — Support Vector Machine
### 8.1 Hyperparameter Tuning (C)

In [49]:
print("Tuning C on validation set...\n")
best_C_svm, best_f1 = 1.0, 0.0

for C in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]:
    svm_tmp = LinearSVC(C=C, class_weight="balanced", max_iter=2000,
                         random_state=42, dual=True)
    svm_tmp.fit(X_tr, y_tr)
    f1_val = f1_score(y_val, svm_tmp.predict(X_val), average="macro")
    marker = " ← best" if f1_val > best_f1 else ""
    print(f"  C={C:<4}  val_f1={f1_val:.4f}{marker}")
    if f1_val > best_f1:
        best_f1, best_C_svm = f1_val, C

print(f"\nBest C : {best_C_svm}  (val F1 = {best_f1:.4f})")

Tuning C on validation set...

  C=0.01  val_f1=0.7331 ← best
  C=0.1   val_f1=0.8448 ← best
  C=0.5   val_f1=0.8933 ← best
  C=1.0   val_f1=0.9027 ← best
  C=2.0   val_f1=0.8987
  C=5.0   val_f1=0.8880

Best C : 1.0  (val F1 = 0.9027)


### 8.2 Train Final SVM

In [50]:
# CalibratedClassifierCV wraps LinearSVC to provide predict_proba (needed for AUC)
svm_base = LinearSVC(C=best_C_svm, class_weight="balanced",
                      max_iter=2000, random_state=42, dual=True)
svm = CalibratedClassifierCV(svm_base, cv=3)

t0 = time.time()
svm.fit(X_train_combined, y_train)
print(f"Train time : {time.time()-t0:.2f}s")

joblib.dump(svm, f"{OUTPUT_DIR}/svm_model.pkl")
print(f"Model saved to {OUTPUT_DIR}/svm_model.pkl")

Train time : 0.50s
Model saved to /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_1/svm_model.pkl


### 8.3 Evaluate SVM on Test Set

In [51]:
res_svm = evaluate("SVM (LinearSVC + Calibrated)", svm, X_test_combined, y_test)

──────────────────────────────────────────────────
  SVM (LinearSVC + Calibrated)
──────────────────────────────────────────────────
  Accuracy   : 0.8793  ✓ (target ≥ 0.85)
  Precision  : 0.8798  (macro)
  Recall     : 0.8793  (macro)
  F1         : 0.8793  ✓ (target ≥ 0.80)
  AUC-ROC    : 0.9295
  Inf. time  : 0.0099 ms/sample

  Confusion Matrix (rows=actual, cols=predicted):
                 Pred Non-CB   Pred CB
  Actual Non-CB       672         78
  Actual CB           103        647

  Classification Report:
               precision    recall  f1-score   support

non-clickbait     0.8671    0.8960    0.8813       750
    clickbait     0.8924    0.8627    0.8773       750

     accuracy                         0.8793      1500
    macro avg     0.8798    0.8793    0.8793      1500
 weighted avg     0.8798    0.8793    0.8793      1500



## 9. Save Artifacts & Results Summary

In [52]:
results = [res_mnb, res_lr, res_svm]
with open(f"{OUTPUT_DIR}/classical_results.json", "w") as f:
    json.dump(results, f, indent=2)

joblib.dump(tfidf,  f"{OUTPUT_DIR}/tfidf_vectorizer.pkl")
joblib.dump(scaler, f"{OUTPUT_DIR}/feature_scaler.pkl")

print("Saved:")
print(f"  {OUTPUT_DIR}/classical_results.json")
print(f"  {OUTPUT_DIR}/tfidf_vectorizer.pkl")
print(f"  {OUTPUT_DIR}/feature_scaler.pkl")

Saved:
  /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_1/classical_results.json
  /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_1/tfidf_vectorizer.pkl
  /content/drive/MyDrive/NLPAOL_V3/outputs/classical_models_1/feature_scaler.pkl


## 10. Comparison Table

In [53]:
print(f"{'Model':<35} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>7} {'ms/s':>8}")
print("─" * 78)
for r in results:
    auc_str = f"{r['auc_roc']:.4f}" if r["auc_roc"] else "  N/A "
    acc_flag = " ✓" if r["accuracy"] >= 0.85 else "  "
    f1_flag  = " ✓" if r["f1_macro"] >= 0.80  else "  "
    print(f"{r['model']:<35} {r['accuracy']:>7.4f}{acc_flag} "
          f"{r['precision_macro']:>7.4f} {r['recall_macro']:>7.4f} "
          f"{r['f1_macro']:>7.4f}{f1_flag} {auc_str:>7} "
          f"{r['inference_ms_per_sample']:>8.4f}")
print()
print("✓ = meets target  |  Target: Accuracy ≥ 0.85, F1 ≥ 0.80")

Model                                   Acc    Prec     Rec      F1     AUC     ms/s
──────────────────────────────────────────────────────────────────────────────
Multinomial Naive Bayes              0.8187    0.8193  0.8187  0.8186 ✓  0.8973   0.0010
Logistic Regression                  0.8793 ✓  0.8793  0.8793  0.8793 ✓  0.9311   0.0009
SVM (LinearSVC + Calibrated)         0.8793 ✓  0.8798  0.8793  0.8793 ✓  0.9295   0.0099

✓ = meets target  |  Target: Accuracy ≥ 0.85, F1 ≥ 0.80
